# 06 Statistical Modelling 1: Linear Regression
This notebook will fit a weighted linear regression model for FHVHV wait time, and predicts raw wait time on a log scale. 

- We may expect this statistical model to be more of an interpretive model than predictive, as it enforces linearity, making feature effects easy to explain, but limits the model's complexity and ability to capture nonlinear relationships. Nevertheless, we will make predictions with this model to compare with the machine learning model evaluation later.
- We will assign a higher weight to alert rows for balance (rows with alert is ~2% of all rows), s.t. the model does not overrepresent for non-alert rows during fitting, as our primary goal is to interpret / predict in the context of when an alert arises. 



In [26]:
# Restart spark if required
try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "America/New_York") # timezone set to NYC
    .getOrCreate()
)
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.adaptive.enabled", "true")

spark.sparkContext.setLogLevel("ERROR") # silence warnings

## Set up and configuration

In [ ]:
# Imports
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

import pandas as pd
import math


In [28]:
# Raw spatial data from `01_fetch_data.ipynb`
taxi_zones_path = "../data/raw/taxi_zones/taxi_zones.shp"

# Input directories from `02_preprocess_data.ipynb`
processed_dir = "../data/processed"

# Input directories from `03_feature_engineering.ipynb`
features_dir = "../data/features"

# Output directories for `05_exploratory_analysis.ipynb`
analysis_dir = "../data/analysis"

In [29]:
sdf_tlc_hourly = spark.read.parquet(f"{analysis_dir}/tlc_zone_hourly_temporal")
sdf_tlc_hourly.show(5)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+---------+----------------+----------+---------+--------+-------------------+----+----------------+-------------------+-----------+-----------+------------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|  borough|       taxi_zone|n_stations|ridership|n_alerts|disruption_severity|coco|weather_severity|              month|hour_of_day|day_of_week|hour_of_week|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+---------+----------------+----------+---------+--------+-------------------+----+----------------+-------------------+-----------+-----------+------------+
|2026-03-26 17:00:00|          94|           0|           0.00|           81|         1873.83|             406.67|    Bronx|   Fordham South|         2|      339|       0|                

## Preparing inputs for `pyspark.ml` linear regression
- build the features vector with numeric and categorical (one-hot-encoded) features
- include temporal (`month_num`, `hour_of_week`) and spatial (`PULocationID`) predictors as we may expect wait time to differs strongly by zone, month, and weekly time, in addition to the alert signal
- train-test split for 2025 vs. 2026 respectively 
- assign higher weight to underrepresented alerts instances (PULocation, hour) to balance for alerts row prediction error
- Let Manhattan be the baseline reference borough for interaction `alert_borough` and `severity_borough`; Manhattan interaction columns are omitted to absorb its effects into the baseline `has_alert` and `disruption_severity`.
- For all other interaction terms, the baseline reference is from `handleInvalid="keep"` which makes it less interpretable. It is mainly used as spatial / temporal controls for this model.

In [30]:
# Build base modelling table
base_df = (
    sdf_tlc_hourly
    .filter(F.col("avg_wait_time_fhvhv").isNotNull())
    .withColumn("year", F.year("hour"))
    .withColumn("month_num", F.date_format("hour", "MM"))
    .withColumn("hour_of_day", F.format_string("%03d", F.col("hour_of_day").cast("int")))
    .withColumn("day_of_week", F.col("day_of_week"))
    .withColumn("hour_of_week", F.format_string("%03d", F.col("hour_of_week").cast("int")))
    .withColumn("PULocationID", F.format_string("%03d", F.col("PULocationID").cast("int")))
    .withColumn("has_alert", (F.col("n_alerts") > 0).cast("int"))
    .withColumn("log_ridership", F.log1p("ridership"))
    .withColumn("label_log_wait", F.log1p("avg_wait_time_fhvhv"))
    # note `disruption_severity` is already in sdf_tlc_hourly

    # Let Manhattan be the baseline reference borough for interaction interpretation
    .withColumn("alert_bronx", F.when(F.col("borough") == "Bronx", F.col("has_alert")).otherwise(0))
    .withColumn("alert_brooklyn", F.when(F.col("borough") == "Brooklyn", F.col("has_alert")).otherwise(0))
    .withColumn("alert_queens", F.when(F.col("borough") == "Queens", F.col("has_alert")).otherwise(0))
    .withColumn("severity_bronx", F.when(F.col("borough") == "Bronx", F.col("disruption_severity")).otherwise(0))
    .withColumn("severity_brooklyn", F.when(F.col("borough") == "Brooklyn", F.col("disruption_severity")).otherwise(0))
    .withColumn("severity_queens", F.when(F.col("borough") == "Queens", F.col("disruption_severity")).otherwise(0))
)


In [31]:
# Train test split 
train_base = base_df.filter(F.col("year") == 2025) # train (12 months)
test_base = base_df.filter(F.col("year") == 2026)  # test (5 months)

# Split into rows with / without alerts
n_alert_train = train_base.filter(F.col("has_alert") == 1).count()
n_no_alert_train = train_base.filter(F.col("has_alert") == 0).count()

# Assign weighting for rows with alert, add to train and test
alert_weight = n_no_alert_train / n_alert_train

train_base = train_base.withColumn(
    "sample_weight",
    F.when(F.col("has_alert") == 1, F.lit(alert_weight)).otherwise(F.lit(1.0))
)

test_base = test_base.withColumn("sample_weight", F.lit(1.0))

# Print for reference
print(f"Alert training rows:    {n_alert_train:,}")
print(f"No-alert training rows: {n_no_alert_train:,}")
print(f"Alert row weight:       {alert_weight:.2f}")


Alert training rows:    34,076
No-alert training rows: 1,293,947
Alert row weight:       37.97


We choose the following predictors to form a linear regression formula like this 
$$\begin{align}
\log(1 + \texttt{avg\_wait\_time}_i) = 
& \ \beta_0 \\
& + \beta_1 * \texttt{log\_ridership}_i \\
& + \beta_2 * \texttt{weather\_severity}_i \\
& + \beta_3 * \texttt{has\_alert}_i \\
& + \beta_4 * \texttt{disruption\_severity}_i \\
& + \sum_{z\in \texttt{PULocationID}} \alpha_{\texttt{PULocationID}, z} * \mathbb I(\texttt{PULocationID}_i = z)\\
& + \sum_{m\in \texttt{month\_num}} \alpha_{\texttt{month\_num}, m} * \mathbb I(\texttt{month\_num}_i = m) \\
& + \sum_{h\in \texttt{hour\_of\_week}} \alpha_{\texttt{hour\_of\_week}, h} * \mathbb I(\texttt{hour\_of\_week}_i = h) \\
& + \delta_{\text{Bronx}} * \texttt{has\_alert}_i * \mathbb{I}(\texttt{borough}_i = \text{Bronx}) \\
& + \delta_{\text{Brooklyn}} * \texttt{has\_alert}_i * \mathbb{I}(\texttt{borough}_i = \text{Brooklyn}) \\
& + \delta_{\text{Queens}} * \texttt{has\_alert}_i * \mathbb{I}(\texttt{borough}_i = \text{Queens}) \\
& + \gamma_{\text{Bronx}} * \texttt{disruption\_severity}_i * \mathbb{I}(\texttt{borough}_i = \text{Bronx}) \\
& + \gamma_{\text{Brooklyn}} * \texttt{disruption\_severity}_i * \mathbb{I}(\texttt{borough}_i = \text{Brooklyn}) \\
& + \gamma_{\text{Queens}} * \texttt{disruption\_severity}_i * \mathbb{I}(\texttt{borough}_i = \text{Queens}) \\
& + \epsilon_i
\end{align}
$$

- $\beta_0$: intercept of linear model 
- $\beta_\texttt{feat}$: coefficients for numerical predictor $\texttt{feat}$ 
- $\alpha_{\texttt{feat}, i}$: coefficient for the $i$ th level in categorical predictor $\texttt{feat}$, with reference to baseline (spatial / temporal controls) $\alpha_{\texttt{feat}, 0}$
- $\delta_{\texttt{borough}}$: coefficient for the interaction between $\texttt{alert}$ and given $\texttt{borough}$ (**additional** effect), with reference to baseline $\beta_3$ ($\texttt{has\_alert}$ in Manhattan)
- $\gamma_{\texttt{borough}}$: coefficient for the interaction between $\texttt{disruption\_severity}$ and given $\texttt{borough}$ (**additional** effect), with reference to baseline $\beta_4$ ($\texttt{disruption\_severity}$ in Manhattan)
- $\epsilon$: error of linear model, assume $\epsilon_i \sim N(0, \sigma^2)$

In [ ]:
# Group predictors to categorical or numerical
categorical_cols = [
    "PULocationID",
    "month_num",
    "hour_of_week"
]

numeric_cols = [
    "n_stations",
    "log_ridership",
    "weather_severity",
    "has_alert",
    "disruption_severity",
    "alert_bronx",
    "alert_brooklyn",
    "alert_queens",
    "severity_bronx",
    "severity_brooklyn",
    "severity_queens",
]

# Turn text categories to numeric categories (e.g. Bronx -> 0, Brooklyn -> 1)
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") # , stringOrderType="alphabetAsc" 
    for col in categorical_cols
] # make unseen bucket by default

# Turn numeric categories to one-hot-encoding binary indicator variables 
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_oh", dropLast=True)
    for col in categorical_cols
] # dropLast column removes unseen bucket; all observed feature columns are retained

# Combine all numeric and categorical (one-hot) categories to features vector for sparkml
assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{col}_oh" for col in categorical_cols],
    outputCol="features"
)


## Fit linear regression model with `pyspark.ml`
- fit regression model with compiled features vector 
- evaluate on both log and raw scales using RMSE, MAE, and $R^2$
- examine the fitted coefficients; particularly interaction term coefficients, which may suggest how wait time differs by month, zone, and weekly time pattern 

In [33]:
# Fit weighted linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label_log_wait",
    predictionCol="pred_log_wait",
    weightCol="sample_weight",
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

model_linreg = pipeline.fit(train_base)

pred_linreg = (
    model_linreg
    .transform(test_base)
    .withColumn("pred_wait_seconds", F.expm1("pred_log_wait"))
)

In [ ]:
# Evaluate on log scale
rmse_log = RegressionEvaluator(labelCol="label_log_wait", predictionCol="pred_log_wait", metricName="rmse").evaluate(pred_linreg)
mae_log = RegressionEvaluator(labelCol="label_log_wait", predictionCol="pred_log_wait", metricName="mae").evaluate(pred_linreg)
r2_log = RegressionEvaluator( labelCol="label_log_wait", predictionCol="pred_log_wait", metricName="r2").evaluate(pred_linreg)

# Evaluate on original seconds scale
rmse_seconds = RegressionEvaluator(labelCol="avg_wait_time_fhvhv", predictionCol="pred_wait_seconds", metricName="rmse").evaluate(pred_linreg)
mae_seconds = RegressionEvaluator(labelCol="avg_wait_time_fhvhv", predictionCol="pred_wait_seconds", metricName="mae").evaluate(pred_linreg)
r2_seconds = RegressionEvaluator(labelCol="avg_wait_time_fhvhv", predictionCol="pred_wait_seconds", metricName="r2").evaluate(pred_linreg)

# Evaluate alert-affected rows separately
alert_pred_linreg = pred_linreg.filter(F.col("has_alert") == 1)
alert_mae_seconds = RegressionEvaluator(labelCol="avg_wait_time_fhvhv", predictionCol="pred_wait_seconds", metricName="mae").evaluate(alert_pred_linreg)
alert_rmse_seconds = RegressionEvaluator(labelCol="avg_wait_time_fhvhv", predictionCol="pred_wait_seconds", metricName="rmse").evaluate(alert_pred_linreg)

print("Linear Regression: log1p(avg_wait_time_fhvhv)")
print(f"RMSE log scale:        {rmse_log:.4f}")
print(f"MAE log scale:         {mae_log:.4f}")
print(f"R2 log scale:          {r2_log:.4f}")
print()
print("Linear Regression: wait time in seconds")
print(f"RMSE seconds:          {rmse_seconds:.2f}")
print(f"MAE seconds:           {mae_seconds:.2f}")
print(f"R2 seconds:            {r2_seconds:.4f}")
print()
print("Alert-affected rows only")
print(f"Alert-hour RMSE sec:   {alert_rmse_seconds:.2f}")
print(f"Alert-hour MAE sec:    {alert_mae_seconds:.2f}")
print(f"Alert rows:            {alert_pred_linreg.count():,}")

Linear Regression: log1p(avg_wait_time_fhvhv)
RMSE log scale:        0.2481
MAE log scale:         0.1805
R2 log scale:          0.1881

Linear Regression: wait time in seconds
RMSE seconds:          97.71
MAE seconds:           60.38
R2 seconds:            0.1262

Alert-affected rows only
Alert-hour RMSE sec:   100.94
Alert-hour MAE sec:    59.80
Alert rows:            16,063


In [35]:
# Extract fitted coefficients
fitted_lr = model_linreg.stages[-1]

feature_attrs = (
    model_linreg
    .transform(train_base.limit(1))
    .schema["features"]
    .metadata["ml_attr"]["attrs"]
)

feature_names = []
for attr_type in feature_attrs:
    feature_names += [attr["name"] for attr in feature_attrs[attr_type]]

coef_linreg = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": fitted_lr.coefficients.toArray()
    })
    .sort_values("coefficient", ascending=False)
)

coef_linreg

,feature,coefficient
162,month_num_oh_04,0.684669
70,PULocationID_oh_144,0.619646
158,month_num_oh_03,0.598737
54,PULocationID_oh_108,0.572166
332,n_stations,0.343414
...,...,...
135,PULocationID_oh_088,-0.216206
104,PULocationID_oh_223,-0.223637
125,PULocationID_oh_250,-0.243739
57,PULocationID_oh_114,-0.251879


The extracted coefficients have highest absolute coefficients in categorical one-hot encoded columns from `PULocationID` and `month_num`. This suggests that the temporal and spatial features explain more wait-time variation than simply subway alerts / disruption severity level, which is very plausible given alert rows are only 2% of the data, and in the real life context, wait time is indeed highly influenced by the WHERE and WHEN, in addition to disruption severity level. 

We can use disruption-related coefficients (interaction) to examine whether disruption features still explain additional wait-time pressure.

In [36]:
# Key disruption-related coefficients
coef_linreg[
    coef_linreg["feature"].str.contains(
        "alert|severity|ridership|weather",
        case=False
    )
].sort_values("feature")

,feature,coefficient
337,alert_bronx,0.163480
338,alert_brooklyn,0.147381
339,alert_queens,0.121242
336,disruption_severity,0.146539
335,has_alert,0.083075
333,log_ridership,0.074694
340,severity_bronx,0.247792
341,severity_brooklyn,0.181527
342,severity_queens,0.171085
334,weather_severity,0.187816


In [37]:
# Convert severity effects to approximate percentage change by borough.
# `parameter` is the model parameter added at that row:
# - Manhattan: beta_4 = disruption_severity
# - Outer boroughs: gamma_borough = severity_borough additional effect

coef_lookup = dict(zip(coef_linreg["feature"], coef_linreg["coefficient"]))
base_severity = coef_lookup["disruption_severity"]

severity_effect_rows = []

# Understanding interaction between severity and borough 
for borough, extra_col in [
    ("Manhattan", None),
    ("Bronx", "severity_bronx"),
    ("Brooklyn", "severity_brooklyn"),
    ("Queens", "severity_queens"),
]: # for Manhattan, we just have the baseline 
    if extra_col is None:
        parameter_name = "beta_4"
        parameter_value = base_severity
        total_coef = base_severity
    else: # for other boroughs, we have an EXTRA effect
        parameter_name = f"gamma_{borough.lower()}"
        parameter_value = coef_lookup[extra_col]
        total_coef = base_severity + parameter_value

    # the combined coef gives effect for each borough in total
    severity_effect_rows.append({
        "borough": borough,
        "parameter": parameter_name,
        "parameter_value": parameter_value,
        "log_coef_per_severity_unit": total_coef,
        "approx_pct_change": 100 * (math.exp(total_coef) - 1)
    })

severity_effect_pd = pd.DataFrame(severity_effect_rows)
severity_effect_pd

,borough,parameter,parameter_value,log_coef_per_severity_unit,approx_pct_change
0,Manhattan,beta_4,0.146539,0.146539,15.782018
1,Bronx,gamma_bronx,0.247792,0.394331,48.339139
2,Brooklyn,gamma_brooklyn,0.181527,0.328066,38.828095
3,Queens,gamma_queens,0.171085,0.317624,37.386024


The disruption related coefficients give insight into how disruption severity is associated with fhvhv wait time in each borough, with respect to Manhattan (encoded as baseline level in `disruption_severity`, $\beta_4$). The interaction terms for Bronx, Brooklyn, and Queens show additional fitted severity effects relative to Manhattan, clear log-scale increases for Bronx (+24.78), Brooklyn (+18.15) and Queens (+17.11). 
$$
\begin{align}
\beta_4 & = 0.146539 \text{ base slope} \\
\gamma_\text{Bronx} & = 0.247792 \text{ extra slope} \\
\gamma_\text{Brooklyn} & = 0.181527 \text{ extra slope} \\
\gamma_\text{Queens} & = 0.171085 \text{ extra slope} 
\end{align}
$$

We can form the severity slope for each borough with baseline Manhattan 
$$
\begin{align}
\text{Manhattan severity slope} &: \beta_4 & = 0.146539	 \\
\text{Bronx severity slope} &: \beta_4 + \gamma_\text{Bronx} & = 0.394331 \\
\text{Brooklyn severity slope} &: \beta_4 + \gamma_\text{Brooklyn} & = 0.328066 \\
\text{Queens severity slope} &: \beta_4 + \gamma_\text{Queens} & = 0.317624
\end{align}
$$

We will turn the log-scale increases (in wait time with respect to disruption severity, holding all other predictors fixed). Using Bronx as an example, for each unit increase in disruption severity, the change in predicted `log1p(wait_time)` is given by $\beta_4 + \gamma_\text{Bronx} = 0.394331$. Using log laws to turn difference into ratio, we can write this as:
$$
\beta_4 + \gamma_\text{Bronx} = 0.394311 = \log(1+\texttt{wait\_time}_\text{new}) - \log(1+\texttt{wait\_time}_\text{old}) = \log\left(\frac{1+\texttt{wait\_time}_\text{new}}{1+\texttt{wait\_time}_\text{old}}\right)
$$

Exponentiate both sides and we find $\exp(\beta_4 + \gamma_\text{Bronx}) = \frac{1+\texttt{wait\_time}_\text{new}}{1+\texttt{wait\_time}_\text{old}}$, which is the multiplicative factor. 

Then we minus 1 to represent proportion increase. e.g. IF $\exp(\beta_4 + \gamma_\text{Bronx})=1.2$, there is a $20\%$ increase from old to new. 

Using the formula above, the percentage change in wait time with respect to disruption severity (holding all other predictors fixed) for each borough is given by:
$$
\begin{align}
\text{Manhattan}&: \exp(\beta_4) - 1 &= \exp(0.146539) - 1 &\approx +15.782018\% \text{ per severity unit} \\
\text{Bronx}&: \exp(\beta_4 + \gamma_\text{Bronx}) - 1 &= \exp(0.146539 + 0.247792) - 1 &\approx +48.339139\% \text{ per severity unit}\\
\text{Brooklyn}&: \exp(\beta_4 + \gamma_\text{Brooklyn}) - 1 &= \exp(0.146539 + 0.181527) - 1 &\approx +38.828095\% \text{ per severity unit}\\
\text{Queens}&: \exp(\beta_4 + \gamma_\text{Queens}) - 1 &= \exp(0.146539 + 0.171085) - 1 &\approx +37.386024\% \text{ per severity unit}
\end{align}
$$
This suggests that, for the same increase in disruption severity, wait times rise more sharply in these outer boroughs than in Manhattan, especially in the Bronx and Brooklyn.